# **Пересдача по СТАТОИВ (Бустинг)**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("default")
sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("data/Exam_Score_Prediction.csv")
df.head()

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        20000 non-null  int64  
 1   age               20000 non-null  int64  
 2   gender            20000 non-null  object 
 3   course            20000 non-null  object 
 4   study_hours       20000 non-null  float64
 5   class_attendance  20000 non-null  float64
 6   internet_access   20000 non-null  object 
 7   sleep_hours       20000 non-null  float64
 8   sleep_quality     20000 non-null  object 
 9   study_method      20000 non-null  object 
 10  facility_rating   20000 non-null  object 
 11  exam_difficulty   20000 non-null  object 
 12  exam_score        20000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 2.0+ MB


In [4]:
df.isnull().sum()

student_id          0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numeric columns — исключаем student_id
num_cols_clean = [c for c in num_cols if c in df_clean.columns and c != "student_id"]

summary = []
outlier_masks = {}

for col in num_cols_clean:
    col_series = df_clean[col]
    s = col_series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # маска выбросов по IQR
    mask_iqr = (col_series < lower) | (col_series > upper)

    # маска выбросов по Z-score (>3)
    mean = s.mean()
    std = s.std(ddof=0)  # population std
    if std == 0 or np.isnan(std):
        mask_z = pd.Series(False, index=col_series.index)
    else:
        mask_z = (np.abs((col_series - mean) / std) > 3)

    mask_any = mask_iqr | mask_z
    outlier_masks[col] = mask_any

    summary.append({
        "col": col,
        "n": len(col_series),
        "n_out_iqr": int(mask_iqr.sum()),
        "n_out_z": int(mask_z.sum()),
        "n_out_any": int(mask_any.sum()),
        "pct_out_any": round(100 * mask_any.sum() / len(col_series), 2)
    })

summary_df = pd.DataFrame(summary).set_index("col").sort_values("pct_out_any", ascending=False)
print("Сводка выбросов (IQR + Z>3):")
print(summary_df)

# Показать примеры выбросов (по каждой колонке — до 5 строк, с контекстом exam_score)
for col in summary_df.index:
    if summary_df.loc[col, "n_out_any"] == 0:
        continue
    print(f"\nПримеры выбросов по '{col}' (top 5):")
    display(df_clean.loc[outlier_masks[col], [col, "exam_score"]].sort_values(col, ascending=False).head(5))

# пометка строк, где есть хоть один числовой выброс
df_clean["_is_outlier_any"] = np.column_stack([outlier_masks[c] for c in num_cols_clean]).any(axis=1)
print(f"\nСтрок с любым числовым выбросом: {df_clean['_is_outlier_any'].sum()} "
      f"({df_clean['_is_outlier_any'].mean()*100:.2f}%)")

# Опционально: быстрые boxplot'ы для каждой числовой колонки
for col in num_cols_clean:
    plt.figure(figsize=(6,2.5))
    df_clean.boxplot(column=col)
    plt.title(f"Boxplot: {col}")
    plt.tight_layout()
    plt.show()


NameError: name 'num_cols' is not defined

# **Prediction Model Build**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
df_clean = df.copy()

cat_cols = [
    "gender", "course", "internet_access",
    "sleep_quality", "study_method",
    "facility_rating", "exam_difficulty"
]

num_cols = [
    "student_id", "age", "study_hours",
    "class_attendance", "sleep_hours"
]

from sklearn.preprocessing import LabelEncoder

le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    le_dict[col] = le

X = df_clean.drop("exam_score", axis=1)
y = df_clean["exam_score"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    random_state=42
)
rf.fit(X_train, y_train)

xgb = XGBRegressor(
    n_estimators=350,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
xgb.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print(f"Random Forest → R²: {rf_r2:.4f}, MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}")
print(f"XGBoost       → R²: {xgb_r2:.4f}, MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# 3 класса по квантилям
# 0 = low, 1 = mid, 2 = high
y_cls = pd.qcut(df_clean["exam_score"], q=3, labels=[0, 1, 2]).astype(int)

X_clf = X.copy()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

rf_clf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
rf_clf.fit(X_train_c, y_train_c)

xgb_clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    eval_metric="mlogloss",
    random_state=42
)
xgb_clf.fit(X_train_c, y_train_c)

for name, model in [("RandomForest", rf_clf), ("XGBoost", xgb_clf)]:
    preds = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds, average="macro")
    print(f"{name}  —  accuracy: {acc:.3f},  f1_macro: {f1:.3f}")
    print(classification_report(y_test_c, preds))
    print("confusion_matrix:\n", confusion_matrix(y_test_c, preds))
    print("-" * 40)


df_test_clf = X_test_c.copy()
df_test_clf["exam_score_true"] = y_test_c
df_test_clf["rf_pred_class"] = rf_clf.predict(X_test_c)
df_test_clf["xgb_pred_class"] = xgb_clf.predict(X_test_c)